# AI Risk Model Evaluation Notebook

This notebook validates model quality using **train/test data**, cross-validation summaries, and visual diagnostics to detect overfitting or underfitting.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.src.model_quality import regression_metrics, diagnose_fit

sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path.cwd()
ML_DIR = NOTEBOOK_DIR.parent
MODELS_DIR = ML_DIR / "models"
PROCESSED_DIR = ML_DIR / "data" / "processed"

MODEL_BUNDLE_PATH = MODELS_DIR / "risk_model_v1.joblib"
METADATA_PATH = MODELS_DIR / "metadata.json"
TRAIN_DATA_PATH = PROCESSED_DIR / "risk_data_train.csv"
TEST_DATA_PATH = PROCESSED_DIR / "risk_data_test.csv"

: 

In [ ]:
required_paths = [MODEL_BUNDLE_PATH, METADATA_PATH, TRAIN_DATA_PATH, TEST_DATA_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing artifacts. Run ml.src.train first. Missing: {missing}")

bundle = joblib.load(MODEL_BUNDLE_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)

feature_names = bundle["feature_names"]
model = bundle["model"]
scaler = bundle.get("scaler")

x_train = train_df[feature_names]
y_train = train_df["risk_score"]
x_test = test_df[feature_names]
y_test = test_df["risk_score"]

x_train_input = scaler.transform(x_train) if scaler is not None else x_train
x_test_input = scaler.transform(x_test) if scaler is not None else x_test

train_pred = model.predict(x_train_input)
test_pred = model.predict(x_test_input)

print("Model version:", metadata.get("version"))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))

In [ ]:
train_metrics = regression_metrics(y_train.to_numpy(), train_pred)
test_metrics = regression_metrics(y_test.to_numpy(), test_pred)
fit_quality = diagnose_fit(train_metrics, test_metrics)
cv_summary = metadata.get("cross_validation", {})

report_df = pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "test", **test_metrics},
])
display(report_df)

print("Cross-validation summary:")
print(cv_summary)
print("Fit quality:")
print(fit_quality)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].scatter(y_train, train_pred, alpha=0.35, s=14, label="Train", color="#1f77b4")
ax[0].plot([0, 1], [0, 1], linestyle="--", color="black")
ax[0].set_title("Train: Actual vs Predicted")
ax[0].set_xlabel("Actual risk score")
ax[0].set_ylabel("Predicted risk score")

ax[1].scatter(y_test, test_pred, alpha=0.45, s=18, label="Test", color="#ff7f0e")
ax[1].plot([0, 1], [0, 1], linestyle="--", color="black")
ax[1].set_title("Test: Actual vs Predicted")
ax[1].set_xlabel("Actual risk score")
ax[1].set_ylabel("Predicted risk score")

plt.tight_layout()
plt.show()

In [ ]:
train_residuals = y_train.to_numpy() - train_pred
test_residuals = y_test.to_numpy() - test_pred

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train_residuals, kde=True, stat="density", color="#1f77b4", ax=ax[0], label="Train", alpha=0.5)
sns.histplot(test_residuals, kde=True, stat="density", color="#ff7f0e", ax=ax[0], label="Test", alpha=0.5)
ax[0].set_title("Residual Distribution")
ax[0].set_xlabel("Residual (actual - predicted)")
ax[0].legend()

ax[1].scatter(test_pred, test_residuals, alpha=0.45, color="#d62728")
ax[1].axhline(0.0, linestyle="--", color="black")
ax[1].set_title("Test Residuals vs Predicted")
ax[1].set_xlabel("Predicted")
ax[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
if hasattr(model, "feature_importances_"):
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=importance, x="importance", y="feature", color="#2ca02c")
    plt.title("Feature Importance (Random Forest)")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    display(importance)
else:
    print("Model does not expose feature_importances_.")

In [ ]:
summary = {
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
    "cross_validation": cv_summary,
    "fit_quality": fit_quality,
}
print(json.dumps(summary, indent=2))